In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma



In [1]:
import pandas as pd
books = pd.read_csv("books_cleaned.csv")
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web - A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5083,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5084,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5085,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5086,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that - Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [2]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5083    9788172235222 On A Train Journey Home To North...
5084    9788173031014 This book tells the tale of a ma...
5085    9788179921623 Wisdom to Create a Life of Passi...
5086    9788185300535 This collection of the timeless ...
5087    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5088, dtype: object

In [17]:
books["tagged_description"].to_csv("tagged_description.txt",
                                   sep="\n",
                                   header=False,
                                   index=False)

In [7]:
raw_documents = TextLoader("tagged_description.txt",  encoding='utf-8').load()
text_splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1168, which is longer than the specified 0
Created a chunk of size 1214, which is longer than the specified 0
Created a chunk of size 373, which is longer than the specified 0
Created a chunk of size 309, which is longer than the specified 0
Created a chunk of size 483, which is longer than the specified 0
Created a chunk of size 482, which is longer than the specified 0
Created a chunk of size 960, which is longer than the specified 0
Created a chunk of size 188, which is longer than the specified 0
Created a chunk of size 843, which is longer than the specified 0
Created a chunk of size 296, which is longer than the specified 0
Created a chunk of size 197, which is longer than the specified 0
Created a chunk of size 881, which is longer than the specified 0
Created a chunk of size 1088, which is longer than the specified 0
Created a chunk of size 1189, which is longer than the specified 0
Created a chunk of size 304, which is longer than the specified 0
Create

In [9]:
documents[88]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780060004507 An adaptation of the original story follows the activities of seven children in nineteenth-century New England as they prepare for the Thanksgiving holiday while Mother is away caring for Grandmother.')

In [10]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db_books = Chroma.from_documents(
    documents,
    embedding=embedding_model
)

In [11]:
query = " a book to teach children about roman history"
docs = db_books.similarity_search(query, k=10)
docs

[Document(id='f3b42b02-fbb8-4b65-8845-0a4aaa42fa10', metadata={'source': 'tagged_description.txt'}, page_content="9780192833006 Cornelius Tacitus, Rome's greatest historian, was inspired to take up his pen when the assassination of Domitian ended `fifteen years of enforced silence'. Agricola is the biography of his late father-in-law and an account of Roman Britain. Germania gives insight into Rome's most dangerous enemies, the Germans, and is the only surviving specimen from the ancient world of an ethnographic study. Each in its way has had immense influence on our perception of Rome and the northern `barbarians' and the edition reflects recent research in Roman-British and Roman-German history. - ;`Long may the barbarians continue, I pray, if not to love us, at least to hate one another.' Cornelius Tacitus, Rome's greatest historian and the last great writer of classical Latin prose, produced his first two books in AD 98. He was inspired to take up his pen when the assassination of 

In [12]:
books[books["isbn13"] ==int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
945,9780192833006,0192833006,Agricola and Germany,Tacitus,Literary Collections,http://books.google.com/books/content?id=wS7IJ...,"Cornelius Tacitus, Rome's greatest historian, ...",1999.0,3.98,224.0,25.0,Agricola and Germany,"9780192833006 Cornelius Tacitus, Rome's greate..."


In [15]:
def retrieve_semantic_recommendationv(
        query: str,
        top_k: int = 10) -> pd.DataFrame:
        recs = db_books.similarity_search(query, k = 50)

        book_list = []
        for i in range (0 , len(recs)):
            book_list+=[int(recs[i].page_content.strip('"' ).split()[0])]

        return books[books["isbn13"].isin(book_list)].head(top_k)


In [16]:
retrieve_semantic_recommendationv("lesbian romance", 5)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
155,9780060574215,0060574216,"Men Are from Mars, Women Are from Venus",John Gray,Family & Relationships,http://books.google.com/books/content?id=MUw_d...,Rediscover the most famous relationship book e...,2004.0,3.54,368.0,126108.0,"Men Are from Mars, Women Are from Venus - The ...",9780060574215 Rediscover the most famous relat...
177,9780060598846,0060598840,Love Overboard,Janet Evanovich,Fiction,http://books.google.com/books/content?id=5pClJ...,"Dear Reader: In a previous life, before the ti...",2005.0,3.65,244.0,11223.0,Love Overboard,"9780060598846 Dear Reader: In a previous life,..."
209,9780060753634,0060753633,Mating in Captivity,Esther Perel,Psychology,http://books.google.com/books/content?id=-HIhM...,A guide for loving couples who are looking to ...,2006.0,4.13,272.0,7699.0,Mating in Captivity - Reconciling the Erotic a...,9780060753634 A guide for loving couples who a...
259,9780060854096,006085409X,Dates From Hell,Kim Harrison;Lynsay Sands;Kelley Armstrong;Lor...,Fiction,http://books.google.com/books/content?id=D07pn...,She thought her date was out of this world. Ac...,2006.0,3.78,416.0,10467.0,Dates From Hell,9780060854096 She thought her date was out of ...
282,9780060916466,006091646X,The Dance of Intimacy,Harriet Lerner,Psychology,http://books.google.com/books/content?id=tTKc8...,"In The Dance of Intimacy, the bestselling auth...",1990.0,4.06,255.0,7128.0,The Dance of Intimacy - A Woman's Guide to Cou...,"9780060916466 In The Dance of Intimacy, the be..."


In [17]:
from transformers import pipeline
fiction_categories = ["fiction" "non-fiction "]


pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)


Device set to use cpu


In [19]:
books.loc[books["simple_categories"] == "fiction", "description"].reset_index(drop=True)[0]

KeyError: 'simple_categories'